# SafeTrade — DPO fine-tune บน H100 (LiCO)

notebook นี้เป็นตัวสั่งงาน สำหรับคลัสเตอร์ที่เข้าได้ทาง Jupyter อย่างเดียว ไม่มี SSH/sbatch

**ก่อนเริ่ม** — ต้องมีไฟล์เหล่านี้จากเครื่องตัวเองแล้ว (push ขึ้น git หรืออัปโหลดผ่านแถบไฟล์ของ Jupyter):
- `train_dpo.py`, `requirements-dpo.txt`
- `output/dpo/` ทั้งโฟลเดอร์ (`dpo_pairs_train.jsonl`, `dpo_pairs_eval.jsonl`, `system_prompt.txt`, `dpo_manifest.json`)

**รันทีละ cell ตามลำดับ** — cell 1-4 ทำครั้งเดียวต่อ session ใหม่

## 1. สำรวจเครื่องก่อน

ดูว่าได้ GPU อะไร เหลือ disk เท่าไหร่ และ **session นี้มีเวลากี่ชั่วโมง** (สำคัญ — เทรนเต็มใช้ ~3 ชม.)

In [ ]:
!nvidia-smi
!echo '--- disk ---' && df -h $HOME
!echo '--- python ---' && python -V && which python

## 2. ตั้ง path ที่ไม่หายตอน session จบ

`HF_HOME` สำคัญมาก — ถ้าไม่ตั้ง จะโหลด base model 16GB ใหม่ทุกครั้งที่เปิด session ซึ่งกินเวลาไปฟรีๆ

`PROJECT` ตั้งไว้ให้ตรงกับการ `git clone` ลง home แล้ว (home ของ LiCO อยู่บน NFS ไฟล์ไม่หายตอนปิดเครื่อง) ถ้าวางไฟล์ไว้ที่อื่นให้แก้บรรทัดนี้

In [ ]:
import os
from pathlib import Path

PROJECT = Path.home() / "Project-AI" / "backend" / "scripts"   # << ตรงกับ git clone ลง home
VENV = PROJECT / "venv-dpo"
OUTDIR = PROJECT / "dpo_out"

os.environ["HF_HOME"] = str(PROJECT / "hf")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

OUTDIR.mkdir(parents=True, exist_ok=True)
print("PROJECT :", PROJECT)
print("HF_HOME :", os.environ["HF_HOME"])
print("มีไฟล์ครบไหม:")
for f in ["train_dpo.py", "requirements-dpo.txt", "output/dpo/dpo_pairs_train.jsonl", "output/dpo/system_prompt.txt"]:
    print("  ", "✅" if (PROJECT / f).exists() else "❌", f)

## 3. ติดตั้ง Python 3.11 + ไลบรารี ลง venv แยก

image ของ LiCO มาพร้อม Python 3.8 ซึ่งเก่าเกินไป (torch/transformers/trl ต้องการ 3.9 ขึ้นไป) จึงใช้ **uv** ติดตั้ง Python 3.11 ไว้ในโฟลเดอร์ของเราเอง ไม่ต้องใช้สิทธิ์ admin

**ต้องแยก venv** — ไม่ลงทับของ image และห้าม `pip install` แล้ว `import` ใน kernel เดียวกัน ให้เรียกผ่าน `!` เสมอ

`--index-strategy unsafe-best-match` จำเป็น: index ของ PyTorch มี numpy/typing-extensions เวอร์ชันเก่าติดมาด้วย ถ้าไม่ใส่ uv จะหยิบจากที่นั่นอย่างเดียวแล้ว resolve ไม่ผ่าน

(ครั้งแรก ~5-10 นาที รอบต่อไปข้ามเองเพราะ venv อยู่บน storage ถาวร)

**เช็คบรรทัดสุดท้ายต้องขึ้น `cuda True`** — ถ้าขึ้น `cuda False` แปลว่า driver ของเครื่อง (525 / CUDA 12.0) ใช้กับ PyTorch แบบ CUDA 12.4 ไม่ได้ ให้แก้บรรทัด `--extra-index-url` ใน `requirements-dpo.txt` จาก `cu124` เป็น `cu118` แล้วลบโฟลเดอร์ `venv-dpo` รัน cell นี้ใหม่

In [ ]:
UV = Path.home() / ".local" / "bin" / "uv"
if not UV.exists():
    !curl -LsSf https://astral.sh/uv/install.sh | sh
print("uv:", UV, "✅" if UV.exists() else "❌ ติดตั้งไม่สำเร็จ")

!test -d {VENV} || {UV} venv --python 3.11 {VENV}
!{UV} pip install --python {VENV}/bin/python --index-strategy unsafe-best-match -r {PROJECT}/requirements-dpo.txt
!{VENV}/bin/python -c "import sys, torch, transformers, trl, peft; print('python', sys.version.split()[0]); print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'); print('transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__)"

## 4. โหลด base model ล่วงหน้า

แยกเป็น cell ของตัวเอง เพราะถ้าเน็ตหลุดกลางทางจะได้ไม่เสียรอบเทรนไปด้วย (~16GB)

In [ ]:
!{VENV}/bin/huggingface-cli download Qwen/Qwen3-8B --exclude "original/*"
!du -sh $HF_HOME

## 5. Dry run — วัดความยาว token ก่อนเทรนจริง

ใช้แค่ tokenizer ไม่โหลดโมเดล ไม่กิน GPU (~1 นาที)

**ดูตัวเลขที่พิมพ์ออกมาให้ดี** ถ้า `max_length` เกิน ~9000 ให้หยุดแล้วตรวจข้อมูลก่อน
เพราะ DPOConfig ตั้ง default ไว้แค่ 1024 ซึ่งจะตัด prompt ทิ้งเงียบๆ

In [ ]:
!cd {PROJECT} && {VENV}/bin/python train_dpo.py --dry-run --output-dir {OUTDIR}

In [ ]:
# ดูว่า prompt ที่ render ออกมาหน้าตาถูกไหม
# ต้องลงท้ายด้วย: '/no_think /no_think<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
# (ตรงกับที่ Ollama ส่งจริงเมื่อเรียกด้วย think=false)
txt = (OUTDIR / "render_sample0.txt").read_text(encoding="utf-8")
print(txt[:300])
print("\n... ตัดกลาง ...\n")
print(repr(txt[-200:]))

## 6. เทรนจริง

ใช้ `nohup ... &` เพื่อให้งานรอดถ้า kernel restart หรือเผลอกด interrupt
(แต่ไม่รอดถ้า scheduler ตัด session — นั่นคือเหตุผลที่สคริปต์เซฟ checkpoint ทุก 25 step และ resume เองได้)

ถ้าเวลา session ไม่ถึง 4 ชม. ให้ใส่ `--epochs 1` ก่อน

In [ ]:
!cd {PROJECT} && nohup {VENV}/bin/python -u train_dpo.py --output-dir {OUTDIR} > {PROJECT}/train.log 2>&1 &
print("เริ่มเทรนแล้ว — รัน cell ข้างล่างเพื่อดู log")

In [ ]:
# รัน cell นี้ซ้ำได้เรื่อยๆ เพื่อดูความคืบหน้า
!tail -40 {PROJECT}/train.log
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv

### สิ่งที่ต้องจับตาใน log

| ค่า | ปกติ | ผิดปกติ |
|---|---|---|
| `rewards/accuracies` | ค่อยๆ ขึ้นไป 0.75-0.95 | พุ่งไป ~1.0 ตั้งแต่ step 20 → โมเดลแยกคู่ได้จากรูปแบบข้อความ ไม่ใช่ทักษะตรวจจับ |
| `rewards/margins` | เพิ่มขึ้นเรื่อยๆ | นิ่งใกล้ 0 → learning rate ต่ำไป |
| `logps/chosen` | นิ่งหรือขึ้นเล็กน้อย | ร่วงพร้อม `logps/rejected` → โมเดลกำลังพังทั้งกระดาน ให้เพิ่ม `--rpo-alpha 2.0` |

ถ้า session ตายกลางทาง: เปิดใหม่ รัน cell 2-3 แล้วสั่งเทรนซ้ำ — สคริปต์จะหา checkpoint ล่าสุดเองอัตโนมัติ

## 7. Merge adapter เข้า base model

⚠️ **ดาวน์โหลด `dpo_out/final_adapter` กลับเครื่องก่อนทำขั้นนี้** — ไฟล์เล็ก (~80MB) แต่สร้างใหม่ไม่ได้ถ้าหาย
ส่วนที่เหลือ (merged, gguf) สร้างซ้ำได้เสมอ

In [ ]:
merge_code = f'''
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-8B", torch_dtype=torch.bfloat16)
merged = PeftModel.from_pretrained(base, "{OUTDIR}/final_adapter").merge_and_unload()
merged.save_pretrained("{OUTDIR}/merged", safe_serialization=True, max_shard_size="4GB")
AutoTokenizer.from_pretrained("{OUTDIR}/final_adapter").save_pretrained("{OUTDIR}/merged")
print("merged เสร็จ")
'''
(PROJECT / "_merge.py").write_text(merge_code)
!{VENV}/bin/python {PROJECT}/_merge.py

## 8. แปลงเป็น GGUF

`convert_hf_to_gguf.py` เป็น Python ล้วน ไม่ต้องคอมไพล์ llama.cpp

แปลงบนคลัสเตอร์เพื่อให้ไฟล์ที่ต้องโหลดกลับเล็กลง แล้วค่อยไป quantize ต่อที่เครื่องตัวเอง
ด้วย llama.cpp build สำเร็จรูป (`llama-quantize` เป็นไบนารีที่ต้องคอมไพล์ อย่าพยายามบิลด์บนคลัสเตอร์)

In [ ]:
# image ของ LiCO ไม่มี git — โหลด llama.cpp เป็น tarball แทน
!test -d {PROJECT}/llama.cpp || (mkdir -p {PROJECT}/llama.cpp && curl -sL https://github.com/ggml-org/llama.cpp/archive/refs/heads/master.tar.gz | tar xz --strip-components=1 -C {PROJECT}/llama.cpp)
# ใช้ gguf จาก source ของ llama.cpp เอง — ตัวบน PyPI มักตามสคริปต์แปลงไม่ทัน
!{UV} pip install --python {VENV}/bin/python {PROJECT}/llama.cpp/gguf-py
!{VENV}/bin/python {PROJECT}/llama.cpp/convert_hf_to_gguf.py {OUTDIR}/merged --outfile {OUTDIR}/safetrade-f16.gguf --outtype f16
!ls -lh {OUTDIR}/*.gguf

## 9. ขั้นตอนต่อที่เครื่องตัวเอง

ดาวน์โหลด `safetrade-f16.gguf` ผ่านแถบไฟล์ของ Jupyter (คลิกขวา → Download) แล้วที่เครื่องตัวเอง:

```bash
# 1. quantize ทั้งสองแบบ แล้ววัดทั้งคู่ (Q4 อาจกลบผลของ LoRA ได้)
llama-quantize safetrade-f16.gguf safetrade-Q8_0.gguf Q8_0
llama-quantize safetrade-f16.gguf safetrade-Q4_K_M.gguf Q4_K_M

# 2. คัด Modelfile ของ base มาทั้งไฟล์ — ต้องได้ TEMPLATE, stop token (<|im_end|>), top_k 20 มาครบ
#    ไม่งั้น prompt จะไม่ตรงกับตอนเทรน หรือโมเดลไม่รู้จักหยุด
ollama show --modelfile qwen3:8b > Modelfile

# 3. แก้ไฟล์ Modelfile สองจุด:
#    - เปลี่ยนบรรทัด FROM เป็น   FROM ./safetrade-Q8_0.gguf
#    - เติมท้ายไฟล์             PARAMETER num_ctx 8192   (ไม่งั้นบั๊ก prompt ถูกตัดกลับมา)
ollama create safetrade-qwen3:8b -f Modelfile

# 4. baseline บน holdout ชุดเดียวกัน (ไม่ต้องรันโมเดลใหม่ ใช้ผลเดิมกรองเอา)
python grade_results.py --only-conversations output/dpo/holdout_conversations.json \
    --output output/grading_baseline_holdout.json

# 5. วัดโมเดลที่ fine-tune แล้ว บน holdout เดียวกัน
OLLAMA_MODEL=safetrade-qwen3:8b python blind_test.py \
    --only-conversations output/dpo/holdout_conversations.json \
    --output output/blind_test_finetuned.json --fresh
python grade_results.py --input output/blind_test_finetuned.json \
    --output output/grading_finetuned.json
```

หมายเหตุ: `ollama_client.py` ส่ง `think=false` อยู่แล้ว ซึ่ง `train_dpo.py` เทรนด้วยรูป prompt แบบเดียวกันนี้ — **ถ้า Ollama อัปเดตระหว่างนี้ ต้องตรวจ parity ใหม่**